# Exercise Prediction – Final Pipeline

Load pose data from `data/raw/train.csv`, train XGBoost and PyTorch models, pick best, save to `models/`.

In [ ]:
import os
import sys
import numpy as np
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_loader import load_raw_data, ensure_directories
from src.preprocessing import prepare_data, compute_class_weights
from src.train import build_and_train_xgboost, build_and_train_pytorch
from src.predict import save_model_and_artifacts

In [ ]:
ensure_directories('.')
df, data_dir = load_raw_data('.')
print('Data shape:', df.shape)
print('Pose distribution:')
print(df['pose'].value_counts())

In [ ]:
X_train, X_val, y_train, y_val, encoder, scaler, class_names, feature_cols = prepare_data(df)
class_weights = compute_class_weights(y_train)
input_size = X_train.shape[1]
num_classes = len(class_names)
print('Train:', X_train.shape, 'Val:', X_val.shape)
print('Classes:', class_names)
print('Feature columns:', len(feature_cols))

In [ ]:
# Train XGBoost
xgb_model, xgb_acc = build_and_train_xgboost(X_train, y_train, X_val, y_val, num_classes)
print(f'XGBoost val accuracy: {xgb_acc:.4f}')

In [ ]:
# Train PyTorch MLP
pt_model, pt_acc = build_and_train_pytorch(
    X_train, y_train, X_val, y_val,
    input_size, num_classes,
    class_weights=class_weights,
    epochs=40
)
print(f'PyTorch val accuracy: {pt_acc:.4f}')

In [ ]:
# Pick best model and save
if xgb_acc >= pt_acc:
    best_model = xgb_model
    best_acc = xgb_acc
    model_type = 'xgboost'
else:
    best_model = pt_model
    best_acc = pt_acc
    model_type = 'pytorch'

print(f'Best: {model_type} (acc={best_acc:.4f})')
save_model_and_artifacts(
    best_model, encoder, scaler, class_names, best_acc,
    models_dir='models', model_type=model_type, input_size=input_size,
    feature_columns=feature_cols
)
print('Model saved to models/ (scaler.pkl, label_encoder.pkl, metadata.json)')